# How to read one of the zarr stores and grab a stack item

In [42]:
import xarray as xr
from pystac_client import Client
import rasterio
from rasterio.plot import show
import planetary_computer as pc

Pretty straight forward with xarray. These zarr stores are also zarr v3. So I imagine you need that installed in your environment

In [38]:
zarr_path = '../lake_detection_binary_masks_2018.zarr'
lake_zarr = xr.open_dataset(zarr_path)

/home/m484s199/miniconda3/envs/py312/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:99: UserWarning: The codec `vlen-bytes` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


The dataset has a bit of text in the attributes. The main two variables are `ndwi_mask` and `source_items`.

`ndwi_mask`: A binary datacube of NDWI>0.3

`source_items`: The stac_ids used in the `ndwi_mask`

In [39]:
lake_zarr

<xarray.Dataset> Size: 150GB
Dimensions:       (x: 26088, time: 109, y: 52904, stac_item: 9)
Coordinates:
  * x             (x) float64 209kB -2.577e+05 -2.577e+05 ... 3.205e+03
  * time          (time) datetime64[ns] 872B 2018-05-05 ... 2018-09-27
  * y             (y) float64 423kB -1.944e+06 -1.944e+06 ... -2.473e+06
Dimensions without coordinates: stac_item
Data variables:
    spatial_ref   int64 8B ...
    ndwi_mask     (time, y, x) int8 150GB ...
    source_items  (time, stac_item) object 8kB ...
Attributes:
    crs:                       EPSG:3413
    description:               ~Daily NDWI masks clipped to Greenland ice she...
    resolution_m:              10
    cloud_percentage_filter:   10%
    ice_sheet_masks:           https://www.doi.org/10.5067/579TO87M7IZB
    ice_sheet_masks_citation:  Greene, C.A., Gardner, A.S., Wood, M. et al. U...

Now to grab a stac item from `source_items` and read an image. The `source_items` variable as all the ids that were used in the merge

In [43]:
from pystac_client import Client

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",modifier=pc.sign_inplace)
item_id = lake_zarr['source_items'].isel(time=2).data[0].decode('utf-8') # make sure to decode the string. Not sure the best way to store these
collection = catalog.get_collection("sentinel-2-l2a")
item = collection.get_item(item_id) 
item

<Item id=S2B_MSIL2A_20180510T152909_R111_T22WEC_20201026T092714>

Read the bands with rasterio (can be xarray too). This will read the whole image. Can add an window for a subset read which will be much faster

In [ ]:
with rasterio.open(item.assets["B04"].href) as src:
    red = src.read(1)

with rasterio.open(item.assets["B03"].href) as src:
    green = (src.read(1))
    
with rasterio.open(item.assets["B02"].href) as src:
    blue = (src.read(1))

rgb_stack = np.stack([red, green, blue])

a quick viz

In [ ]:
# Let rasterio handle the scaling
show(rgb_stack, adjust=True)